# Data Warehouse ETL

Dit notebook laadt data van SDM naar DWH en detecteert eerst welk DWH-schema actief is in `BikeToDrive_6_Datawarehouse.db`:
- **Oud schema**: alle dimensies en feiten gebruiken natural keys
- **Nieuw schema**: SCD Type 1 voor vaste dimensies en SCD Type 2 voor Klant en Monteur
- **Feiten**: Accessoire_Verkoop, Fiets_Verkoop, Onderhoud

**SDM-inlaadstrategie**: het SDM-notebook gebruikt een **full refresh**: eerst alle SDM-tabellen leegmaken, daarna alles opnieuw inladen.

**Let op**: draai dit notebook van boven naar beneden. Dimensies eerst, dan feiten.

In [ ]:
# 1. IMPORTS EN INSTELLINGEN
import sys
import sqlite3
import pandas as pd
from pathlib import Path
from loguru import logger

log_path = Path("../logs/dwh_load.log")
log_path.parent.mkdir(parents=True, exist_ok=True)
logger.remove()
logger.add(
    sys.stderr,
    level="INFO",
)
logger.add(
    log_path,
    level="INFO",
    encoding="utf-8",
    backtrace=True,
    diagnose=False,
)

DB_SDM = "../WEEK 3/BikeToDrive_6 _SourceDataModel.db"
DB_DWH = "BikeToDrive_6_Datawarehouse.db"
VANDAAG = pd.Timestamp.now().strftime("%Y-%m-%d")

logger.info(f"Imports gedaan — SDM: {DB_SDM}, DWH: {DB_DWH}")

INFO | Imports gedaan — SDM: ../WEEK 3/BikeToDrive_6 _SourceDataModel.db, DWH: BikeToDrive_6_Datawarehouse.db


In [ ]:
# 2. ETL CONFIGURATIE
# Detecteer eerst welk DWH-schema actief is in de huidige database.

def detect_dwh_strategy(db_path):
    with sqlite3.connect(db_path) as conn:
        klant_cols = {row[1] for row in conn.execute("PRAGMA table_info(Klant)")}
        monteur_cols = {row[1] for row in conn.execute("PRAGMA table_info(Monteur)")}
        accessoire_verkoop_cols = {
            row[1] for row in conn.execute("PRAGMA table_info(Accessoire_Verkoop)")
        }

    uses_scd2 = (
        {"klant_sk", "begintijd", "eindtijd"}.issubset(klant_cols)
        and {"monteur_sk", "begintijd", "eindtijd"}.issubset(monteur_cols)
        and "klant_sk" in accessoire_verkoop_cols
        and "monteur_sk" in accessoire_verkoop_cols
    )
    return "scd2" if uses_scd2 else "scd1"


DWH_STRATEGIE = detect_dwh_strategy(DB_DWH)

BASE_SCD1_DIMS = [
    ("Datum", "datum", """
        SELECT datum FROM Accessoire_Verkoop
        UNION SELECT datum FROM Fiets_Verkoop
        UNION SELECT datum FROM Onderhoud
    """),
    ("Filiaal", "filiaalnr", """
        SELECT filiaalnr, naam, adres, provincie FROM Accessoire_Verkoop_Filiaal
        UNION SELECT filiaalnr, naam, adres, provincie FROM Fiets_Verkoop_Filiaal
        UNION SELECT filiaalnr, naam, adres, provincie FROM Onderhoud_Filiaal
    """),
    ("Leverancier", "leveranciernr", """
        SELECT leveranciernr, naam, adres, woonplaats FROM Accessoire_Verkoop_Leverancier
        UNION SELECT leveranciernr, naam, adres, woonplaats FROM Accessoire_Inkoop_Leverancier
    """),
    ("Accessoire", "accessoirenr", """
        SELECT accessoirenr, naam, soort, standaardprijs, inkoopprijs FROM Accessoire_Verkoop_Accessoire
        UNION SELECT accessoirenr, naam, soort, standaardprijs, inkoopprijs FROM Accessoire_Inkoop_Accessoire
    """),
    ("Fabrikant", "fabrikantnr", """
        SELECT fabrikantnr, naam, adres, plaats FROM Fiets_Verkoop_Fabrikant
        UNION SELECT fabrikantnr, naam, adres, plaats FROM Onderhoud_Fabrikant
        UNION SELECT fabrikantnr, naam, adres, plaats FROM Fiets_Inkoop_Fabrikant
    """),
    ("Fiets", "fietsnr", """
        SELECT fietsnr, soort, merk, type, kleur, standaardprijs, inkoopprijs FROM Fiets_Verkoop_Fiets
        UNION SELECT fietsnr, soort, merk, type, kleur, standaardprijs, inkoopprijs FROM Onderhoud_Fiets
        UNION SELECT fietsnr, soort, merk, type, kleur, standaardprijs, inkoopprijs FROM Fiets_Inkoop_Fiets
    """),
]

OPTIONELE_SCD1_DIMS = [
    ("Klant", "klantnr", """
        SELECT klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Accessoire_Verkoop_Klant
        UNION SELECT klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Fiets_Verkoop_Klant
    """),
    ("Monteur", "monteurnr", """
        SELECT monteurnr, naam, woonplaats, uurloon FROM Accessoire_Verkoop_Monteur
        UNION SELECT monteurnr, naam, woonplaats, uurloon FROM Fiets_Verkoop_Monteur
        UNION SELECT monteurnr, naam, woonplaats, uurloon FROM Onderhoud_Monteur
    """),
]

SCD2_DIMS = [
    ("Klant", "klantnr", """
        SELECT klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Accessoire_Verkoop_Klant
        UNION SELECT klantnr, naam, adres, woonplaats, geslacht, geboortedatum FROM Fiets_Verkoop_Klant
    """),
    ("Monteur", "monteurnr", """
        SELECT monteurnr, naam, woonplaats, uurloon FROM Accessoire_Verkoop_Monteur
        UNION SELECT monteurnr, naam, woonplaats, uurloon FROM Fiets_Verkoop_Monteur
        UNION SELECT monteurnr, naam, woonplaats, uurloon FROM Onderhoud_Monteur
    """),
] if DWH_STRATEGIE == "scd2" else []

SCD1_DIMS = BASE_SCD1_DIMS + ([] if DWH_STRATEGIE == "scd2" else OPTIONELE_SCD1_DIMS)

FEITEN = [
    ("Accessoire_Verkoop", "accessoire_verkoopnr", """
        SELECT av.accessoire_verkoopnr, av.datum,
               av.klant AS klantnr, av.monteur AS monteurnr,
               m.filiaal AS filiaalnr, av.accessoire AS accessoirenr,
               a.leverancier AS leveranciernr, av.aantal,
               a.standaardprijs, av.verkoopprijs, a.inkoopprijs
        FROM Accessoire_Verkoop av
        JOIN Accessoire_Verkoop_Accessoire a ON av.accessoire = a.accessoirenr
        JOIN Accessoire_Verkoop_Monteur m ON av.monteur = m.monteurnr
    """),
    ("Fiets_Verkoop", "fiets_verkoopnr", """
        SELECT fv.fiets_verkoopnr, fv.datum,
               fv.klant AS klantnr, fv.monteur AS monteurnr,
               m.filiaal AS filiaalnr, fv.fiets AS fietsnr,
               f.fabrikant AS fabrikantnr, fv.aantal,
               f.standaardprijs, fv.verkoopprijs, f.inkoopprijs
        FROM Fiets_Verkoop fv
        JOIN Fiets_Verkoop_Fiets f ON fv.fiets = f.fietsnr
        JOIN Fiets_Verkoop_Monteur m ON fv.monteur = m.monteurnr
    """),
    ("Onderhoud", "onderhoudnr", """ 
        SELECT o.onderhoudnr, o.datum,
               o.monteur AS monteurnr, m.filiaal AS filiaalnr,
               o.fiets AS fietsnr, f.fabrikant AS fabrikantnr,
               o.starttijd, o.eindtijd, m.uurloon
        FROM Onderhoud o
        JOIN Onderhoud_Fiets f ON o.fiets = f.fietsnr
        JOIN Onderhoud_Monteur m ON o.monteur = m.monteurnr
    """),
]

ALLE_TABELLEN = [t for t, _, _ in SCD1_DIMS] + [t for t, _, _ in SCD2_DIMS] + [t for t, _, _ in FEITEN]

logger.info(f"DWH-strategie gedetecteerd: {DWH_STRATEGIE.upper()}")
logger.info(f"Config: {len(SCD1_DIMS)} SCD1, {len(SCD2_DIMS)} SCD2, {len(FEITEN)} feiten")

INFO | DWH-strategie gedetecteerd: SCD2
INFO | Config: 6 SCD1, 2 SCD2, 3 feiten


In [ ]:
# 3. GENERIEKE FUNCTIES

def get_table_columns(conn, tabel):
    return {row[1] for row in conn.execute(f"PRAGMA table_info({tabel})")}


def afleiden(df, tabel):
    # Voeg afgeleide kolommen toe per dimensie en verwijder ruwe meetwaarden
    if tabel == "Datum":
        dt = pd.to_datetime(df["datum"])
        df["jaar"], df["maand"] = dt.dt.year, dt.dt.month
        df["kwartaal"] = (dt.dt.month - 1) // 3 + 1
        df["datum"] = dt.dt.strftime("%Y-%m-%d")
    elif tabel == "Klant":
        leeftijd = (pd.Timestamp.now() - pd.to_datetime(df["geboortedatum"])).dt.days // 365
        df["leeftijdsklasse"] = pd.cut(
            leeftijd,
            bins=[0, 18, 30, 50, 65, 200],
            labels=["Jong", "Jongvolwassen", "Middelbaar", "Senior", "65+"],
        ).astype(str)
    elif tabel == "Monteur":
        df["loonklasse"] = pd.cut(
            df["uurloon"],
            bins=[0, 15, 25, 35, float("inf")],
            labels=["Laag", "Midden", "Hoog", "Top"],
        ).astype(str)
        df = df.drop(columns=["uurloon"])
    elif tabel in ("Accessoire", "Fiets"):
        bins = [0, 10, 25, 50, float("inf")] if tabel == "Accessoire" else [0, 500, 1000, 2000, float("inf")]
        df["prijsklasse"] = pd.cut(
            df["standaardprijs"],
            bins=bins,
            labels=["Budget", "Midden", "Premium", "Luxe"],
        ).astype(str)
        df = df.drop(columns=["standaardprijs", "inkoopprijs"])
    return df


def vergelijk(df_src, df_dwh, pk):
    # Detecteer nieuwe en gewijzigde rijen t.o.v. DWH
    if df_dwh.empty:
        return df_src.copy(), pd.DataFrame(columns=df_src.columns)

    nieuwe = df_src[~df_src[pk].isin(df_dwh[pk])].copy()
    bestaand = df_src[df_src[pk].isin(df_dwh[pk])]
    if bestaand.empty:
        return nieuwe, pd.DataFrame(columns=df_src.columns)

    merged = bestaand.merge(df_dwh, on=pk, suffixes=("_src", "_dwh"))
    data_cols = [c for c in df_src.columns if c != pk]
    verschilt = pd.Series(False, index=merged.index)
    for col in data_cols:
        verschilt |= merged[f"{col}_src"].astype(str) != merged[f"{col}_dwh"].astype(str)

    gewijzigd = df_src[df_src[pk].isin(merged.loc[verschilt, pk])].copy()
    return nieuwe, gewijzigd


def load_scd1(df_src, dwh_conn, tabel, pk):
    # SCD1: insert nieuw, overschrijf gewijzigd
    df_dwh = pd.read_sql_query(f"SELECT * FROM {tabel}", dwh_conn)
    nieuwe, gewijzigd = vergelijk(df_src, df_dwh, pk)

    if not nieuwe.empty:
        nieuwe.to_sql(tabel, dwh_conn, if_exists="append", index=False)
    for _, row in gewijzigd.iterrows():
        cols = [c for c in df_src.columns if c != pk]
        sets = ", ".join(f"{c} = ?" for c in cols)
        dwh_conn.execute(
            f"UPDATE {tabel} SET {sets} WHERE {pk} = ?",
            [row[c] for c in cols] + [row[pk]],
        )

    dwh_conn.commit()
    logger.info(f"  ✓ {tabel}: {len(nieuwe)} nieuw, {len(gewijzigd)} gewijzigd")


def load_scd2(df_src, dwh_conn, tabel, bk):
    # SCD2: sluit oud af (eindtijd), insert nieuwe versie
    sk = f"{tabel.lower()}_sk"
    df_dwh = pd.read_sql_query(f"SELECT * FROM {tabel} WHERE eindtijd IS NULL", dwh_conn)
    src_cols = list(df_src.columns)
    df_dwh_cmp = df_dwh[src_cols] if not df_dwh.empty else pd.DataFrame(columns=src_cols)
    nieuwe, gewijzigd = vergelijk(df_src, df_dwh_cmp, bk)

    if not nieuwe.empty:
        ins = nieuwe.assign(begintijd=VANDAAG, eindtijd=None)
        ins.to_sql(tabel, dwh_conn, if_exists="append", index=False)

    for _, row in gewijzigd.iterrows():
        old_sk = int(df_dwh.loc[df_dwh[bk] == row[bk], sk].iloc[0])
        dwh_conn.execute(f"UPDATE {tabel} SET eindtijd = ? WHERE {sk} = ?", [VANDAAG, old_sk])
        new_row = {**row.to_dict(), "begintijd": VANDAAG, "eindtijd": None}
        pd.DataFrame([new_row]).to_sql(tabel, dwh_conn, if_exists="append", index=False)

    dwh_conn.commit()
    logger.info(f"  ✓ {tabel}: {len(nieuwe)} nieuw, {len(gewijzigd)} gewijzigd (SCD2)")


def prepare_fact_keys(df, dwh_conn, tabel):
    # Gebruik surrogate keys alleen als de feit- en dimensietabellen die echt hebben.
    fact_cols = get_table_columns(dwh_conn, tabel)

    if "klant_sk" in fact_cols and "klantnr" in df.columns:
        klant = pd.read_sql_query(
            "SELECT klantnr, klant_sk FROM Klant WHERE eindtijd IS NULL",
            dwh_conn,
        )
        df = df.merge(klant, on="klantnr").drop(columns="klantnr")

    if "monteur_sk" in fact_cols and "monteurnr" in df.columns:
        monteur = pd.read_sql_query(
            "SELECT monteurnr, monteur_sk FROM Monteur WHERE eindtijd IS NULL",
            dwh_conn,
        )
        df = df.merge(monteur, on="monteurnr").drop(columns="monteurnr")

    return df


def load_feit(df, dwh_conn, tabel, pk):
    # Insert alleen nieuwe feitrijen
    bestaand = pd.read_sql_query(f"SELECT {pk} FROM {tabel}", dwh_conn)[pk]
    nieuwe = df[~df[pk].isin(bestaand)]
    if not nieuwe.empty:
        nieuwe.to_sql(tabel, dwh_conn, if_exists="append", index=False)
    dwh_conn.commit()
    logger.info(f"  ✓ {tabel}: {len(nieuwe)} nieuwe rijen")


logger.info("Generieke functies gedefinieerd")

INFO | Generieke functies gedefinieerd


In [ ]:
# 4. ETL UITVOEREN

logger.info("=" * 50)
logger.info("START ETL: SDM → DWH")
logger.info("=" * 50)

try:
    with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
        dwh.execute("PRAGMA foreign_keys = ON")

        logger.info("STAP 1: Dimensies SCD Type 1")
        for tabel, pk, sql in SCD1_DIMS:
            df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=pk)
            df = afleiden(df, tabel)
            load_scd1(df, dwh, tabel, pk)

        if SCD2_DIMS:
            logger.info("STAP 2: Dimensies SCD Type 2")
            for tabel, bk, sql in SCD2_DIMS:
                df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=bk)
                df = afleiden(df, tabel)
                load_scd2(df, dwh, tabel, bk)
        else:
            logger.info("STAP 2: Overgeslagen — huidige DWH gebruikt geen SCD Type 2 schema")

        logger.info("STAP 3: Feittabellen")
        for tabel, pk, sql in FEITEN:
            df = pd.read_sql_query(sql, sdm)

            if tabel in ("Accessoire_Verkoop", "Fiets_Verkoop"):
                df["omzet"] = df["aantal"] * df["verkoopprijs"]
                df["inkoopwaarde"] = df["aantal"] * df["inkoopprijs"]
                df["brutowinst"] = df["omzet"] - df["inkoopwaarde"]

            elif tabel == "Onderhoud":
                start = pd.to_timedelta(df["starttijd"].astype(str))
                eind = pd.to_timedelta(df["eindtijd"].astype(str))
                df["duur_minuten"] = ((eind - start).dt.total_seconds() / 60).astype(int)
                df["arbeidskosten"] = (df["duur_minuten"] / 60 * df["uurloon"]).round(2)

            df = prepare_fact_keys(df, dwh, tabel)
            load_feit(df, dwh, tabel, pk)

    logger.info("=" * 50)
    logger.info("ETL SUCCESVOL AFGEROND")
    logger.info("=" * 50)

except Exception as e:
    logger.exception(f"FOUT OPGETREDEN: {e}")

INFO | ==================================================
INFO | START ETL: SDM → DWH
INFO | ==================================================
INFO | STAP 1: Dimensies SCD Type 1
INFO |   ✓ Datum: 196 nieuw, 0 gewijzigd
INFO |   ✓ Filiaal: 5 nieuw, 0 gewijzigd
INFO |   ✓ Leverancier: 5 nieuw, 0 gewijzigd
INFO |   ✓ Accessoire: 13 nieuw, 0 gewijzigd
INFO |   ✓ Fabrikant: 11 nieuw, 0 gewijzigd
INFO |   ✓ Fiets: 75 nieuw, 0 gewijzigd
INFO | STAP 2: Dimensies SCD Type 2
INFO |   ✓ Klant: 25 nieuw, 0 gewijzigd (SCD2)
INFO |   ✓ Monteur: 15 nieuw, 0 gewijzigd (SCD2)
INFO | STAP 3: Feittabellen
INFO |   ✓ Accessoire_Verkoop: 100 nieuwe rijen
INFO |   ✓ Fiets_Verkoop: 150 nieuwe rijen
INFO |   ✓ Onderhoud: 50 nieuwe rijen
INFO | ==================================================
INFO | ETL SUCCESVOL AFGEROND
INFO | ==================================================


## Reset DWH

In [ ]:
# Reset DWH (alle tabellen leegmaken)

logger.info("START RESET DWH")

try:
    with sqlite3.connect(DB_DWH) as dwh:
        for tabel in reversed(ALLE_TABELLEN):
            try:
                dwh.execute(f"DELETE FROM {tabel}")
                logger.info(f"  ✓ reset: {tabel}")
            except sqlite3.Error as e:
                logger.error(f"  ✗ reset {tabel}: {e}")
        dwh.commit()
    logger.info("RESET VOLTOOID — DWH IS LEEG")
except Exception as e:
    logger.exception(f"FOUT: {e}")

INFO | START RESET DWH
INFO |   ✓ reset: Onderhoud
INFO |   ✓ reset: Fiets_Verkoop
INFO |   ✓ reset: Accessoire_Verkoop
INFO |   ✓ reset: Monteur
INFO |   ✓ reset: Klant
INFO |   ✓ reset: Fiets
INFO |   ✓ reset: Fabrikant
INFO |   ✓ reset: Accessoire
INFO |   ✓ reset: Leverancier
INFO |   ✓ reset: Filiaal
INFO |   ✓ reset: Datum
INFO | RESET VOLTOOID — DWH IS LEEG


## Verificatie

In [ ]:
# 5. VERIFICATIE — Aantal rijen per tabel
logger.info("Verificatie gestart...")

with sqlite3.connect(DB_DWH) as dwh:
    rows = []
    for tabel in ALLE_TABELLEN:
        try:
            count = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {tabel}", dwh).iloc[0, 0]
            rows.append({"Tabel": tabel, "Rijen": count, "Status": "✓" if count > 0 else "⚠ leeg"})
        except Exception as e:
            rows.append({"Tabel": tabel, "Rijen": "–", "Status": f"✗ {e}"})

pd.DataFrame(rows)

INFO | Verificatie gestart...


,Tabel,Rijen,Status
0,Datum,0,⚠ leeg
1,Filiaal,0,⚠ leeg
2,Leverancier,0,⚠ leeg
3,Accessoire,0,⚠ leeg
4,Fabrikant,0,⚠ leeg
5,Fiets,0,⚠ leeg
6,Klant,0,⚠ leeg
7,Monteur,0,⚠ leeg
8,Accessoire_Verkoop,0,⚠ leeg
9,Fiets_Verkoop,0,⚠ leeg


In [ ]:
# 6. PREVIEW — Bekijk eerste rijen van een DWH tabel
PREVIEW_TABLE = "Klant"

with sqlite3.connect(DB_DWH) as dwh:
    df_preview = pd.read_sql_query(f"SELECT * FROM {PREVIEW_TABLE} LIMIT 10", dwh)

logger.info(f"Preview '{PREVIEW_TABLE}' — {len(df_preview)} rijen")
df_preview

INFO | Preview 'Klant' — 0 rijen


,klant_sk,klantnr,naam,adres,woonplaats,geslacht,geboortedatum,leeftijdsklasse,begintijd,eindtijd


## Test 1 — Toevoegen (nieuwe rij)

Voeg een nieuwe Filiaal toe aan het SDM → herlaad ETL → controleer dat het in DWH verschijnt.

In [ ]:
# TEST 1: Nieuwe rij toevoegen (SCD1 — Filiaal)
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    # VOOR — Filiaal 999 bestaat niet
    print("VOOR:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 999", dwh).to_string())

    # Toevoegen in SDM
    sdm.execute("INSERT INTO Accessoire_Verkoop_Filiaal VALUES (999, 'Test Filiaal', 'Teststraat 1', 'Zuid-Holland')")
    sdm.commit()

    # Herlaad SCD1
    df = pd.read_sql_query(SCD1_DIMS[1][2], sdm).drop_duplicates(subset="filiaalnr")
    load_scd1(afleiden(df, "Filiaal"), dwh, "Filiaal", "filiaalnr")

    # NA — Filiaal 999 moet nu bestaan
    print("NA:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 999", dwh).to_string())

    # Opruimen
    sdm.execute("DELETE FROM Accessoire_Verkoop_Filiaal WHERE filiaalnr = 999"); sdm.commit()
    dwh.execute("DELETE FROM Filiaal WHERE filiaalnr = 999"); dwh.commit()
    logger.info("Opgeruimd")

INFO |   ✓ Filiaal: 6 nieuw, 0 gewijzigd
INFO | Opgeruimd


VOOR: Empty DataFrame
Columns: [filiaalnr, naam, adres, provincie]
Index: []
NA:    filiaalnr          naam         adres     provincie
0        999  Test Filiaal  Teststraat 1  Zuid-Holland


## Test 2 — Wijzigen SCD Type 1 (Update/Overschrijven)

Wijzig een Filiaal-naam in het SDM → herlaad ETL → controleer dat de oude rij is **overschreven**.

In [ ]:
# TEST 2: Wijzigen SCD1 (Filiaal naam overschrijven)
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    # VOOR
    print("VOOR:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 1", dwh).to_string())

    # Bewaar origineel en wijzig SDM
    origineel = pd.read_sql_query("SELECT naam FROM Accessoire_Verkoop_Filiaal WHERE filiaalnr = 1", sdm).iloc[0, 0]
    sdm.execute("UPDATE Accessoire_Verkoop_Filiaal SET naam = 'TEST_SCD1' WHERE filiaalnr = 1"); sdm.commit()

    # Herlaad SCD1
    df = pd.read_sql_query(SCD1_DIMS[1][2], sdm).drop_duplicates(subset="filiaalnr")
    load_scd1(afleiden(df, "Filiaal"), dwh, "Filiaal", "filiaalnr")

    # NA — naam moet overschreven zijn
    print("NA:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 1", dwh).to_string())

    # Herstel origineel
    sdm.execute("UPDATE Accessoire_Verkoop_Filiaal SET naam = ? WHERE filiaalnr = 1", [origineel]); sdm.commit()
    df = pd.read_sql_query(SCD1_DIMS[1][2], sdm).drop_duplicates(subset="filiaalnr")
    load_scd1(afleiden(df, "Filiaal"), dwh, "Filiaal", "filiaalnr")
    logger.info("Hersteld naar origineel")

INFO |   ✓ Filiaal: 0 nieuw, 0 gewijzigd
INFO |   ✓ Filiaal: 0 nieuw, 0 gewijzigd
INFO | Hersteld naar origineel


VOOR:    filiaalnr                 naam              adres      provincie
0          1  BikeWorld Amsterdam  Prinsengracht 100  Noord-Holland
NA:    filiaalnr                 naam              adres      provincie
0          1  BikeWorld Amsterdam  Prinsengracht 100  Noord-Holland


## Test 3 — Wijzigen klantdimensie

Dit testblok past zich aan aan de actieve DWH-strategie:
- Bij **SCD Type 2** krijgt de oude versie een `eindtijd` en komt er een nieuwe actuele versie bij
- Bij **SCD Type 1** wordt de bestaande klant gewoon overschreven

In [ ]:
# TEST 3: Wijzigen klantdimensie volgens actief DWH-schema
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    klant_sql = SCD2_DIMS[0][2] if DWH_STRATEGIE == "scd2" else OPTIONELE_SCD1_DIMS[0][2]
    dwh_klant_sql = (
        "SELECT * FROM Klant WHERE klantnr = 1 ORDER BY klant_sk"
        if DWH_STRATEGIE == "scd2"
        else "SELECT * FROM Klant WHERE klantnr = 1"
    )

    df_klant = pd.read_sql_query(klant_sql, sdm).drop_duplicates(subset="klantnr")
    df_klant = afleiden(df_klant, "Klant")

    df_voor = pd.read_sql_query(dwh_klant_sql, dwh)
    if df_voor.empty:
        logger.info("Klant 1 niet aanwezig in DWH — initialiseer eerst de klantdimensie voor deze test")
        if DWH_STRATEGIE == "scd2":
            load_scd2(df_klant, dwh, "Klant", "klantnr")
        else:
            load_scd1(df_klant, dwh, "Klant", "klantnr")
        df_voor = pd.read_sql_query(dwh_klant_sql, dwh)

    if df_voor.empty:
        raise RuntimeError("Klant 1 bestaat niet in SDM of kon niet in DWH worden geladen.")

    print("VOOR:", df_voor.to_string())
    ORIGINELE_KLANT_WOONPLAATS = df_voor.iloc[0]["woonplaats"]

    bron_tabellen = ["Accessoire_Verkoop_Klant", "Fiets_Verkoop_Klant"]
    for bron_tabel in bron_tabellen:
        sdm.execute(
            f"UPDATE {bron_tabel} SET woonplaats = 'TEST_KLANT' WHERE klantnr = 1"
        )
    sdm.commit()

    df_klant = pd.read_sql_query(klant_sql, sdm).drop_duplicates(subset="klantnr")
    df_klant = afleiden(df_klant, "Klant")

    if DWH_STRATEGIE == "scd2":
        load_scd2(df_klant, dwh, "Klant", "klantnr")
        print(
            "NA:",
            pd.read_sql_query(dwh_klant_sql, dwh).to_string(),
        )
        logger.info("SCD2 gedetecteerd — oude versie afgesloten en nieuwe versie toegevoegd")
    else:
        load_scd1(df_klant, dwh, "Klant", "klantnr")
        print(
            "NA:",
            pd.read_sql_query(dwh_klant_sql, dwh).to_string(),
        )
        logger.info("SCD1 gedetecteerd — klant is overschreven in het DWH")

    logger.info("Brondata is aangepast voor de vervolgtest met een nieuw feit")

INFO | Klant 1 niet aanwezig in DWH — initialiseer eerst de klantdimensie voor deze test
INFO |   ✓ Klant: 25 nieuw, 0 gewijzigd (SCD2)
INFO |   ✓ Klant: 0 nieuw, 1 gewijzigd (SCD2)
INFO | SCD2 gedetecteerd — oude versie afgesloten en nieuwe versie toegevoegd
INFO | Brondata is aangepast voor de vervolgtest met een nieuw feit


VOOR:    klant_sk  klantnr        naam          adres    woonplaats geslacht geboortedatum leeftijdsklasse   begintijd eindtijd
0        26        1  Jan Jansen  Kerkstraat 12  HERLAAD_STAD        M    1985-03-22      Middelbaar  2026-04-08     None
NA:    klant_sk  klantnr        naam          adres    woonplaats geslacht geboortedatum leeftijdsklasse   begintijd    eindtijd
0        26        1  Jan Jansen  Kerkstraat 12  HERLAAD_STAD        M    1985-03-22      Middelbaar  2026-04-08  2026-04-08
1        51        1  Jan Jansen  Kerkstraat 12    TEST_KLANT        M    1985-03-22      Middelbaar  2026-04-08         NaN


## Test 4 — Verwijderen + herstel via ETL

Verwijder een Filiaal uit het DWH → herlaad ETL → controleer dat de rij **terugkomt** vanuit het SDM.

In [ ]:
# TEST 4: Verwijderen uit DWH en herstel via ETL
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    # VOOR
    print("VOOR:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 1", dwh).to_string())

    # Verwijder uit DWH (FK tijdelijk uit voor test-doeleinden)
    dwh.execute("PRAGMA foreign_keys = OFF")
    dwh.execute("DELETE FROM Filiaal WHERE filiaalnr = 1"); dwh.commit()
    dwh.execute("PRAGMA foreign_keys = ON")
    print("NA DELETE:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 1", dwh).to_string())

    # Herlaad SCD1 — filiaal komt terug vanuit SDM
    df = pd.read_sql_query(SCD1_DIMS[1][2], sdm).drop_duplicates(subset="filiaalnr")
    load_scd1(afleiden(df, "Filiaal"), dwh, "Filiaal", "filiaalnr")

    # Hersteld
    print("NA ETL:", pd.read_sql_query("SELECT * FROM Filiaal WHERE filiaalnr = 1", dwh).to_string())

INFO |   ✓ Filiaal: 1 nieuw, 0 gewijzigd


VOOR:    filiaalnr                 naam              adres      provincie
0          1  BikeWorld Amsterdam  Prinsengracht 100  Noord-Holland
NA DELETE: Empty DataFrame
Columns: [filiaalnr, naam, adres, provincie]
Index: []
NA ETL:    filiaalnr                 naam              adres      provincie
0          1  BikeWorld Amsterdam  Prinsengracht 100  Noord-Holland


## Test 5 — Nieuw feit met juiste sleutel

Dit testblok controleert welke sleutel het feit krijgt op basis van het actieve schema:
- Bij **SCD Type 2** moet het feit de actuele surrogate key (`klant_sk`) krijgen
- Bij **SCD Type 1** blijft het feit de natural key (`klantnr`) gebruiken

In [ ]:
# TEST 5: Nieuw feit krijgt de juiste klant-sleutel voor het actieve DWH-schema
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    if "ORIGINELE_KLANT_WOONPLAATS" not in globals():
        raise RuntimeError("Draai eerst Test 3 zodat de originele klantwaarde bekend is.")

    if pd.read_sql_query("SELECT COUNT(*) AS n FROM Datum", dwh).iloc[0, 0] == 0:
        logger.info("DWH is leeg — laad eerst de benodigde dimensies voor deze test")
        for tabel, pk, sql in SCD1_DIMS:
            df_dim = pd.read_sql_query(sql, sdm).drop_duplicates(subset=pk)
            df_dim = afleiden(df_dim, tabel)
            load_scd1(df_dim, dwh, tabel, pk)

        for tabel, bk, sql in SCD2_DIMS:
            df_dim = pd.read_sql_query(sql, sdm).drop_duplicates(subset=bk)
            df_dim = afleiden(df_dim, tabel)
            load_scd2(df_dim, dwh, tabel, bk)

    test_nr = None
    try:
        if DWH_STRATEGIE == "scd2":
            print(
                "Klant versies:",
                pd.read_sql_query(
                    "SELECT klant_sk, klantnr, woonplaats, begintijd, eindtijd FROM Klant WHERE klantnr = 1 ORDER BY klant_sk",
                    dwh,
                ).to_string(),
            )
        else:
            print(
                "Klant rij:",
                pd.read_sql_query("SELECT * FROM Klant WHERE klantnr = 1", dwh).to_string(),
            )

        test_datum = pd.read_sql_query(
            "SELECT datum FROM Datum ORDER BY datum LIMIT 1",
            dwh,
        ).iloc[0, 0]
        max_nr = pd.read_sql_query(
            "SELECT COALESCE(MAX(accessoire_verkoopnr), 0) AS m FROM Accessoire_Verkoop",
            sdm,
        ).iloc[0, 0]
        test_nr = int(max_nr) + 1
        sdm.execute(
            "INSERT INTO Accessoire_Verkoop VALUES (?, ?, 2, 15.00, 1, 1, 1)",
            [test_nr, test_datum],
        )
        sdm.commit()

        df = pd.read_sql_query(FEITEN[0][2], sdm)
        df["omzet"] = df["aantal"] * df["verkoopprijs"]
        df["inkoopwaarde"] = df["aantal"] * df["inkoopprijs"]
        df["brutowinst"] = df["omzet"] - df["inkoopwaarde"]
        df = prepare_fact_keys(df, dwh, "Accessoire_Verkoop")
        load_feit(df, dwh, "Accessoire_Verkoop", "accessoire_verkoopnr")

        print(
            "Feit:",
            pd.read_sql_query(
                f"SELECT * FROM Accessoire_Verkoop WHERE accessoire_verkoopnr = {test_nr}",
                dwh,
            ).to_string(),
        )
    finally:
        if test_nr is not None:
            sdm.execute(
                "DELETE FROM Accessoire_Verkoop WHERE accessoire_verkoopnr = ?",
                [test_nr],
            )
            dwh.execute(
                "DELETE FROM Accessoire_Verkoop WHERE accessoire_verkoopnr = ?",
                [test_nr],
            )
            sdm.commit()
            dwh.commit()

        bron_tabellen = ["Accessoire_Verkoop_Klant", "Fiets_Verkoop_Klant"]
        for bron_tabel in bron_tabellen:
            sdm.execute(
                f"UPDATE {bron_tabel} SET woonplaats = ? WHERE klantnr = 1",
                [ORIGINELE_KLANT_WOONPLAATS],
            )
        sdm.commit()

        klant_sql = SCD2_DIMS[0][2] if DWH_STRATEGIE == "scd2" else OPTIONELE_SCD1_DIMS[0][2]
        df_klant = pd.read_sql_query(klant_sql, sdm).drop_duplicates(subset="klantnr")
        df_klant = afleiden(df_klant, "Klant")

        if DWH_STRATEGIE == "scd2":
            load_scd2(df_klant, dwh, "Klant", "klantnr")
            logger.info("Opgeruimd + klant hersteld via SCD2")
        else:
            load_scd1(df_klant, dwh, "Klant", "klantnr")
            logger.info("Opgeruimd + klant hersteld via SCD1")

INFO | DWH is leeg — laad eerst de benodigde dimensies voor deze test
INFO |   ✓ Datum: 196 nieuw, 0 gewijzigd
INFO |   ✓ Filiaal: 0 nieuw, 0 gewijzigd
INFO |   ✓ Leverancier: 5 nieuw, 0 gewijzigd
INFO |   ✓ Accessoire: 13 nieuw, 0 gewijzigd
INFO |   ✓ Fabrikant: 11 nieuw, 0 gewijzigd
INFO |   ✓ Fiets: 75 nieuw, 0 gewijzigd
INFO |   ✓ Klant: 0 nieuw, 0 gewijzigd (SCD2)
INFO |   ✓ Monteur: 15 nieuw, 0 gewijzigd (SCD2)
INFO |   ✓ Accessoire_Verkoop: 101 nieuwe rijen


Klant versies:    klant_sk  klantnr    woonplaats   begintijd    eindtijd
0        26        1  HERLAAD_STAD  2026-04-08  2026-04-08
1        51        1    TEST_KLANT  2026-04-08         NaN


INFO |   ✓ Klant: 0 nieuw, 1 gewijzigd (SCD2)
INFO | Opgeruimd + klant hersteld via SCD2


Feit:    accessoire_verkoopnr       datum  klant_sk  monteur_sk  filiaalnr  accessoirenr  leveranciernr  aantal  standaardprijs  verkoopprijs  inkoopprijs  omzet  inkoopwaarde  brutowinst
0                   101  2024-01-02        51          16          1             1              1       2           14.99          15.0          8.5   30.0          17.0        13.0


## Test 6 — Herlaad DWH na SDM-wijziging

Simulatie: wijzig data in het SDM en draai de **volledige ETL** opnieuw. Controleer dat:
- **SCD Type 1** dimensie (Leverancier) de wijziging **overschrijft**
- **SCD Type 2** dimensie (Klant) — afhankelijk van het actieve schema — correct wordt bijgewerkt
- Bestaande feiten **ongewijzigd** blijven (geen duplicaten)

In [ ]:
# TEST 6: Herlaad DWH na SDM-wijziging — volledige ETL-loop
with sqlite3.connect(DB_SDM) as sdm, sqlite3.connect(DB_DWH) as dwh:
    dwh.execute("PRAGMA foreign_keys = ON")

    # --- Stap 1: Bewaar originelen ---
    orig_lev = pd.read_sql_query(
        "SELECT naam FROM Accessoire_Verkoop_Leverancier WHERE leveranciernr = 1", sdm
    ).iloc[0, 0]
    orig_klant = pd.read_sql_query(
        "SELECT woonplaats FROM Accessoire_Verkoop_Klant WHERE klantnr = 1", sdm
    ).iloc[0, 0]

    # Bewaar feit-aantallen VOOR de herlaad
    av_count_voor = pd.read_sql_query("SELECT COUNT(*) AS n FROM Accessoire_Verkoop", dwh).iloc[0, 0]
    fv_count_voor = pd.read_sql_query("SELECT COUNT(*) AS n FROM Fiets_Verkoop", dwh).iloc[0, 0]
    oh_count_voor = pd.read_sql_query("SELECT COUNT(*) AS n FROM Onderhoud", dwh).iloc[0, 0]

    print("=== VOOR HERLAAD ===")
    print("Leverancier 1:", pd.read_sql_query("SELECT * FROM Leverancier WHERE leveranciernr = 1", dwh).to_string())
    if DWH_STRATEGIE == "scd2":
        print("Klant 1:", pd.read_sql_query(
            "SELECT klant_sk, klantnr, woonplaats, begintijd, eindtijd FROM Klant WHERE klantnr = 1 ORDER BY klant_sk", dwh
        ).to_string())
    else:
        print("Klant 1:", pd.read_sql_query("SELECT * FROM Klant WHERE klantnr = 1", dwh).to_string())

    # --- Stap 2: Wijzig SDM brondata ---
    # SCD1: Leverancier naam wijzigen (in ALLE brontabellen)
    for tbl in ["Accessoire_Verkoop_Leverancier", "Accessoire_Inkoop_Leverancier"]:
        sdm.execute(f"UPDATE {tbl} SET naam = 'TEST_HERLAAD' WHERE leveranciernr = 1")

    # SCD1/SCD2: Klant woonplaats wijzigen (in ALLE brontabellen)
    for tbl in ["Accessoire_Verkoop_Klant", "Fiets_Verkoop_Klant"]:
        sdm.execute(f"UPDATE {tbl} SET woonplaats = 'HERLAAD_STAD' WHERE klantnr = 1")
    sdm.commit()
    logger.info("SDM brondata gewijzigd")

    # --- Stap 3: Draai de volledige ETL opnieuw ---
    logger.info("Herlaad: volledige ETL starten...")
    for tabel, pk, sql in SCD1_DIMS:
        df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=pk)
        df = afleiden(df, tabel)
        load_scd1(df, dwh, tabel, pk)

    for tabel, bk, sql in SCD2_DIMS:
        df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=bk)
        df = afleiden(df, tabel)
        load_scd2(df, dwh, tabel, bk)

    for tabel, pk, sql in FEITEN:
        df = pd.read_sql_query(sql, sdm)
        if tabel in ("Accessoire_Verkoop", "Fiets_Verkoop"):
            df["omzet"] = df["aantal"] * df["verkoopprijs"]
            df["inkoopwaarde"] = df["aantal"] * df["inkoopprijs"]
            df["brutowinst"] = df["omzet"] - df["inkoopwaarde"]
        elif tabel == "Onderhoud":
            start = pd.to_timedelta(df["starttijd"].astype(str))
            eind = pd.to_timedelta(df["eindtijd"].astype(str))
            df["duur_minuten"] = ((eind - start).dt.total_seconds() / 60).astype(int)
            df["arbeidskosten"] = (df["duur_minuten"] / 60 * df["uurloon"]).round(2)
        df = prepare_fact_keys(df, dwh, tabel)
        load_feit(df, dwh, tabel, pk)

    # --- Stap 4: Controleer resultaten ---
    print("\n=== NA HERLAAD ===")

    # SCD1 check: Leverancier moet overschreven zijn
    lev_na = pd.read_sql_query("SELECT * FROM Leverancier WHERE leveranciernr = 1", dwh)
    print("Leverancier 1:", lev_na.to_string())
    assert lev_na.iloc[0]["naam"] == "TEST_HERLAAD", "SCD1 FOUT: Leverancier niet overschreven!"
    logger.info("✓ SCD1 Leverancier correct overschreven")

    # SCD2/SCD1 check: Klant
    if DWH_STRATEGIE == "scd2":
        klant_na = pd.read_sql_query(
            "SELECT klant_sk, klantnr, woonplaats, begintijd, eindtijd FROM Klant WHERE klantnr = 1 ORDER BY klant_sk", dwh
        )
        print("Klant 1:", klant_na.to_string())
        actueel = klant_na[klant_na["eindtijd"].isna()]
        assert len(actueel) == 1, "SCD2 FOUT: verwacht precies 1 actuele klant-versie!"
        assert actueel.iloc[0]["woonplaats"] == "HERLAAD_STAD", "SCD2 FOUT: woonplaats niet bijgewerkt!"
        afgesloten = klant_na[klant_na["eindtijd"].notna()]
        assert len(afgesloten) >= 1, "SCD2 FOUT: oude versie niet afgesloten!"
        logger.info(f"✓ SCD2 Klant correct — {len(afgesloten)} afgesloten, 1 actueel")
    else:
        klant_na = pd.read_sql_query("SELECT * FROM Klant WHERE klantnr = 1", dwh)
        print("Klant 1:", klant_na.to_string())
        assert klant_na.iloc[0]["woonplaats"] == "HERLAAD_STAD", "SCD1 FOUT: Klant niet overschreven!"
        logger.info("✓ SCD1 Klant correct overschreven")

    # Feit check: geen duplicaten — alleen controleren als de tabel al gevuld was
    av_count_na = pd.read_sql_query("SELECT COUNT(*) AS n FROM Accessoire_Verkoop", dwh).iloc[0, 0]
    fv_count_na = pd.read_sql_query("SELECT COUNT(*) AS n FROM Fiets_Verkoop", dwh).iloc[0, 0]
    oh_count_na = pd.read_sql_query("SELECT COUNT(*) AS n FROM Onderhoud", dwh).iloc[0, 0]
    print(f"\nFeit-aantallen: AV {av_count_voor}→{av_count_na}, FV {fv_count_voor}→{fv_count_na}, OH {oh_count_voor}→{oh_count_na}")

    if av_count_voor > 0:
        assert av_count_na == av_count_voor, f"FOUT: Accessoire_Verkoop duplicaten! {av_count_voor} → {av_count_na}"
    else:
        logger.info(f"  Accessoire_Verkoop was leeg, initieel gevuld met {av_count_na} rijen")

    if fv_count_voor > 0:
        assert fv_count_na == fv_count_voor, f"FOUT: Fiets_Verkoop duplicaten! {fv_count_voor} → {fv_count_na}"
    else:
        logger.info(f"  Fiets_Verkoop was leeg, initieel gevuld met {fv_count_na} rijen")

    if oh_count_voor > 0:
        assert oh_count_na == oh_count_voor, f"FOUT: Onderhoud duplicaten! {oh_count_voor} → {oh_count_na}"
    else:
        logger.info(f"  Onderhoud was leeg, initieel gevuld met {oh_count_na} rijen")

    logger.info("✓ Geen feit-duplicaten na herlaad")

    # --- Stap 5: Herstel originele SDM-data ---
    for tbl in ["Accessoire_Verkoop_Leverancier", "Accessoire_Inkoop_Leverancier"]:
        sdm.execute(f"UPDATE {tbl} SET naam = ? WHERE leveranciernr = 1", [orig_lev])
    for tbl in ["Accessoire_Verkoop_Klant", "Fiets_Verkoop_Klant"]:
        sdm.execute(f"UPDATE {tbl} SET woonplaats = ? WHERE klantnr = 1", [orig_klant])
    sdm.commit()

    # Herstel DWH dimensies naar origineel
    for tabel, pk, sql in SCD1_DIMS:
        df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=pk)
        df = afleiden(df, tabel)
        load_scd1(df, dwh, tabel, pk)
    for tabel, bk, sql in SCD2_DIMS:
        df = pd.read_sql_query(sql, sdm).drop_duplicates(subset=bk)
        df = afleiden(df, tabel)
        load_scd2(df, dwh, tabel, bk)

    logger.info("✓ SDM en DWH hersteld naar origineel")
    logger.info("=== TEST 6 GESLAAGD ===")

INFO | SDM brondata gewijzigd
INFO | Herlaad: volledige ETL starten...
INFO |   ✓ Datum: 0 nieuw, 0 gewijzigd
INFO |   ✓ Filiaal: 0 nieuw, 0 gewijzigd
INFO |   ✓ Leverancier: 0 nieuw, 0 gewijzigd
INFO |   ✓ Accessoire: 0 nieuw, 0 gewijzigd
INFO |   ✓ Fabrikant: 0 nieuw, 0 gewijzigd
INFO |   ✓ Fiets: 0 nieuw, 0 gewijzigd
INFO |   ✓ Klant: 0 nieuw, 0 gewijzigd (SCD2)
INFO |   ✓ Monteur: 0 nieuw, 0 gewijzigd (SCD2)


=== VOOR HERLAAD ===
Leverancier 1:    leveranciernr          naam             adres woonplaats
0              1  TEST_HERLAAD  Fietsenstraat 12  Amsterdam
Klant 1:    klant_sk  klantnr    woonplaats   begintijd    eindtijd
0        26        1  HERLAAD_STAD  2026-04-08  2026-04-08
1        51        1    TEST_KLANT  2026-04-08  2026-04-08
2        52        1  HERLAAD_STAD  2026-04-08         NaN


INFO |   ✓ Accessoire_Verkoop: 0 nieuwe rijen
INFO |   ✓ Fiets_Verkoop: 150 nieuwe rijen
INFO |   ✓ Onderhoud: 50 nieuwe rijen
INFO | ✓ SCD1 Leverancier correct overschreven
INFO | ✓ SCD2 Klant correct — 2 afgesloten, 1 actueel



=== NA HERLAAD ===
Leverancier 1:    leveranciernr          naam             adres woonplaats
0              1  TEST_HERLAAD  Fietsenstraat 12  Amsterdam
Klant 1:    klant_sk  klantnr    woonplaats   begintijd    eindtijd
0        26        1  HERLAAD_STAD  2026-04-08  2026-04-08
1        51        1    TEST_KLANT  2026-04-08  2026-04-08
2        52        1  HERLAAD_STAD  2026-04-08         NaN

Feit-aantallen: AV 100→100, FV 0→150, OH 0→50


AssertionError: FOUT: Fiets_Verkoop duplicaten! 0 → 150